# Amazon ML Challenge — Final Low-Memory Submission Pipeline

**This is the replacement notebook.**

It is designed for the **actual TEST files**, not the 10K development run.

### Design goals
- No LightGBM training
- No giant candidate DataFrame
- No full TEST S2/S3 load
- No SQLite database
- No package installation / internet
- Streams TEST S2 and S3 in chunks
- Uses conservative, precision-first exact entity resolution
- Writes the two official TSV outputs
- Validates `matched_entity_ids ⊆ candidate_entity_ids`
- Includes RAM/disk guards

The matching policy deliberately favors precision because the competition uses macro F0.5.


In [ ]:
import os, re, gc, time, shutil, unicodedata, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RUN_START = time.time()

# ------------------------------------------------------------
# Resource policy
# ------------------------------------------------------------
CHUNK_SIZE = 250_000
MAX_PROCESS_RAM_GB = 12.5       # stop before a typical Kaggle kernel gets killed
MIN_FREE_DISK_GB = 3.0          # keep a safety margin

OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook: FINAL LOW-MEMORY SUBMISSION PIPELINE")
print("Chunk size:", f"{CHUNK_SIZE:,}")
print("Output:", OUTPUT_DIR)

def resource_check(label="", hard=True):
    try:
        import psutil
        rss = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        total, used, free = shutil.disk_usage("/kaggle/working")
        free_gb = free / 1024**3
        print(f"[{label}] RAM RSS={rss:.2f} GB | working disk free={free_gb:.2f} GB")
        if hard and rss > MAX_PROCESS_RAM_GB:
            raise MemoryError(
                f"RAM guard stopped the run at {rss:.2f} GB. "
                "This is intentional to prevent an OOM kernel crash."
            )
        if hard and free_gb < MIN_FREE_DISK_GB:
            raise RuntimeError(
                f"Disk guard stopped the run at {free_gb:.2f} GB free."
            )
    except ImportError:
        total, used, free = shutil.disk_usage("/kaggle/working")
        print(f"[{label}] psutil unavailable | working disk free={free/1024**3:.2f} GB")

resource_check("startup", hard=False)


## 1. Locate the official competition files

In [ ]:
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("."),
]

def find_file(filename):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        try:
            for p in root.rglob(filename):
                return p
        except Exception:
            pass
    return None

TEST_FILES = {
    "s1": "test_source1.tsv",
    "s2": "test_source2.tsv",
    "s3": "test_source3.tsv",
}

TEST_PATHS = {k: find_file(v) for k, v in TEST_FILES.items()}

for k, p in TEST_PATHS.items():
    print(f"{k.upper()}: {p}")

missing = [k for k, p in TEST_PATHS.items() if p is None]
if missing:
    raise FileNotFoundError(
        f"Missing TEST file(s): {missing}. "
        "Attach the official competition TEST dataset to the Kaggle notebook."
    )


## 2. Normalization

In [ ]:
LEGAL_SUFFIXES = {
    "incorporated", "inc", "corporation", "corp", "limited", "ltd",
    "llc", "company", "co", "private limited", "pvt ltd"
}

ADDR_REPLACEMENTS = {
    " street ": " st ", " road ": " rd ", " avenue ": " ave ",
    " boulevard ": " blvd ", " drive ": " dr ", " lane ": " ln ",
    " highway ": " hwy ", " apartment ": " apt "
}

def normalize_text(x):
    if x is None:
        return ""
    x = str(x).casefold()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_business_name(x):
    n = normalize_text(x)
    words = n.split()
    # Remove legal suffixes from the end.
    changed = True
    while changed and words:
        changed = False
        if len(words) >= 2 and " ".join(words[-2:]) in LEGAL_SUFFIXES:
            words = words[:-2]
            changed = True
        elif words[-1] in LEGAL_SUFFIXES:
            words = words[:-1]
            changed = True
    return " ".join(words)

def normalize_address(x):
    a = normalize_text(x)
    a = " " + a + " "
    for old, new in ADDR_REPLACEMENTS.items():
        a = a.replace(old, new)
    return a.strip()

def normalize_country(x):
    return normalize_text(x)

print(normalize_business_name("Payne Enterprises LLC"))
print(normalize_address("3315 Fremont Street, Peoria, IL"))


## 3. Read TEST S1 only

S1 is the reference table. We keep only the columns needed to construct compact lookup keys.

In [ ]:
S1_COLS = ["entity_id", "business_name", "business_address", "country"]

s1 = pd.read_csv(
    TEST_PATHS["s1"],
    sep="\t",
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    usecols=S1_COLS,
    engine="c"
)

s1 = s1.drop_duplicates("entity_id").reset_index(drop=True)

print("TEST S1 rows:", f"{len(s1):,}")
print("S1 columns:", list(s1.columns))

resource_check("after S1 load")


## 4. Build compact unique-key indexes

We deliberately keep **only keys that identify one S1 entity**.

Three exact keys are used:

1. country + normalized business name
2. country + normalized address
3. country + normalized business name + normalized address

This prevents common names/addresses from creating enormous candidate explosions.

A target row can become a candidate through any of these exact keys.

In [ ]:
t0 = time.time()

s1["__country"] = s1["country"].map(normalize_country)
s1["__name"] = s1["business_name"].map(normalize_business_name)
s1["__address"] = s1["business_address"].map(normalize_address)

def build_unique_map(df, key_col):
    tmp = df[[key_col, "entity_id"]]
    tmp = tmp[tmp[key_col] != ""]
    counts = tmp.groupby(key_col, sort=False)["entity_id"].size()
    unique_keys = counts[counts == 1].index
    tmp = tmp[tmp[key_col].isin(unique_keys)]
    return dict(zip(tmp[key_col].to_numpy(), tmp["entity_id"].to_numpy()))

s1["__name_key"] = s1["__country"] + "|" + s1["__name"]
s1["__addr_key"] = s1["__country"] + "|" + s1["__address"]
s1["__combo_key"] = s1["__country"] + "|" + s1["__name"] + "|" + s1["__address"]

# Empty keys are never indexed.
name_map = build_unique_map(s1, "__name_key")
addr_map = build_unique_map(s1, "__addr_key")
combo_map = build_unique_map(s1, "__combo_key")

print("Unique name keys :", f"{len(name_map):,}")
print("Unique addr keys :", f"{len(addr_map):,}")
print("Unique combo keys:", f"{len(combo_map):,}")
print("Index build time :", f"{time.time()-t0:.1f}s")

resource_check("after compact indexes")


## 5. Streaming matcher

The target S2/S3 files are never loaded completely.

For every target row we:
- normalize its name/address/country
- look for an exact unique S1 name match
- look for an exact unique S1 address match
- look for an exact unique S1 name+address match
- retain the candidate IDs
- accept a match only when at least one exact strong condition holds

This is intentionally conservative and avoids the LightGBM score-saturation problem from the previous notebook.

In [ ]:
def format_ids(ids):
    if not ids:
        return ""
    return ",".join(sorted(set(ids)))

# candidate_map[s1_id] = set(target_ids)
candidate_map = defaultdict(set)
match_map = defaultdict(set)

stats = {
    "S2_rows": 0,
    "S3_rows": 0,
    "candidate_pairs": 0,
    "matched_pairs": 0,
    "name_matches": 0,
    "address_matches": 0,
    "combo_matches": 0,
}

def process_target_file(path, source_label):
    print("\n" + "="*65)
    print(f"PROCESSING {source_label}: {path}")
    print("="*65)

    row_count = 0
    local_candidates = 0
    local_matches = 0
    started = time.time()

    for chunk_no, chunk in enumerate(pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        usecols=S1_COLS,
        chunksize=CHUNK_SIZE,
        engine="c"
    ), start=1):

        chunk["__country"] = chunk["country"].map(normalize_country)
        chunk["__name"] = chunk["business_name"].map(normalize_business_name)
        chunk["__address"] = chunk["business_address"].map(normalize_address)

        for row in chunk.itertuples(index=False):
            target_id = row.entity_id
            country = row.__country
            name = row.__name
            address = row.__address

            name_key = country + "|" + name
            addr_key = country + "|" + address
            combo_key = country + "|" + name + "|" + address

            s1_name = name_map.get(name_key) if name else None
            s1_addr = addr_map.get(addr_key) if address else None
            s1_combo = combo_map.get(combo_key) if name and address else None

            ids = set()
            if s1_name:
                ids.add(s1_name)
                stats["name_matches"] += 1
            if s1_addr:
                ids.add(s1_addr)
                stats["address_matches"] += 1
            if s1_combo:
                ids.add(s1_combo)
                stats["combo_matches"] += 1

            if not ids:
                continue

            local_candidates += len(ids)
            stats["candidate_pairs"] += len(ids)

            for s1_id in ids:
                candidate_map[s1_id].add(target_id)

                # Exact unique name/address evidence is strong enough for a
                # precision-first submission.
                match_map[s1_id].add(target_id)
                local_matches += 1

        row_count += len(chunk)

        del chunk
        gc.collect()

        if chunk_no == 1 or chunk_no % 4 == 0:
            print(
                f"{source_label} chunk {chunk_no:,} | "
                f"rows {row_count:,} | "
                f"candidates {local_candidates:,} | "
                f"elapsed {time.time()-started:.1f}s"
            )
            resource_check(f"{source_label} chunk {chunk_no}", hard=True)

    stats[source_label + "_rows"] = row_count
    stats["matched_pairs"] += local_matches

    print(
        f"{source_label} COMPLETE | rows={row_count:,} | "
        f"candidates={local_candidates:,} | "
        f"time={time.time()-started:.1f}s"
    )

process_target_file(TEST_PATHS["s2"], "S2")
resource_check("after S2", hard=True)

process_target_file(TEST_PATHS["s3"], "S3")
resource_check("after S3", hard=True)


## 6. Write official outputs

In [ ]:
# Every TEST S1 must have exactly one row.
all_s1_ids = s1["entity_id"].tolist()

candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"
matching_path = OUTPUT_DIR / "matching_results.tsv"

# Stream the output files too; do not construct giant DataFrames.
with open(candidate_path, "w", encoding="utf-8", newline="") as fc, \
     open(matching_path, "w", encoding="utf-8", newline="") as fm:

    fc.write("source1_entity_id\tcandidate_entity_ids\n")
    fm.write("source1_entity_id\tmatched_entity_ids\n")

    for s1_id in all_s1_ids:
        candidates = candidate_map.get(s1_id, set())
        matches = match_map.get(s1_id, set())

        # Safety invariant: final matches MUST be candidates.
        matches = matches & candidates

        fc.write(s1_id + "\t" + format_ids(candidates) + "\n")
        fm.write(s1_id + "\t" + format_ids(matches) + "\n")

print("Wrote:")
print(candidate_path)
print(matching_path)

resource_check("after output writing", hard=True)


## 7. Submission validation

In [ ]:
def parse_id_list(x):
    x = (x or "").strip()
    return set(i for i in x.split(",") if i)

def validate_outputs():
    issues = []

    s1_ids = set(s1["entity_id"])

    # Valid target IDs were not stored globally, so collect them by streaming.
    valid_target_ids = set()

    for source_key in ["s2", "s3"]:
        for chunk in pd.read_csv(
            TEST_PATHS[source_key],
            sep="\t",
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            usecols=["entity_id"],
            chunksize=CHUNK_SIZE,
            engine="c"
        ):
            valid_target_ids.update(chunk["entity_id"].tolist())

    cand = pd.read_csv(
        candidate_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False
    )
    match = pd.read_csv(
        matching_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False
    )

    # Columns
    if list(cand.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        issues.append("candidate_pairs.tsv has incorrect columns")

    if list(match.columns) != ["source1_entity_id", "matched_entity_ids"]:
        issues.append("matching_results.tsv has incorrect columns")

    # One row per S1
    if len(cand) != len(s1_ids):
        issues.append(f"candidate_pairs rows={len(cand):,}, expected={len(s1_ids):,}")

    if len(match) != len(s1_ids):
        issues.append(f"matching_results rows={len(match):,}, expected={len(s1_ids):,}")

    if cand["source1_entity_id"].duplicated().any():
        issues.append("candidate_pairs contains duplicate S1 IDs")

    if match["source1_entity_id"].duplicated().any():
        issues.append("matching_results contains duplicate S1 IDs")

    cand_map = {}
    match_map_check = {}

    for sid, raw in zip(cand["source1_entity_id"], cand["candidate_entity_ids"]):
        if sid not in s1_ids:
            issues.append(f"candidate_pairs contains non-TEST S1 ID: {sid}")
        ids = parse_id_list(raw)
        if any(i not in valid_target_ids for i in ids):
            issues.append(f"candidate_pairs contains invalid target ID for {sid}")
        if len(ids) != len(parse_id_list(raw)):
            issues.append(f"candidate list issue for {sid}")
        cand_map[sid] = ids

    for sid, raw in zip(match["source1_entity_id"], match["matched_entity_ids"]):
        if sid not in s1_ids:
            issues.append(f"matching_results contains non-TEST S1 ID: {sid}")
        ids = parse_id_list(raw)
        if any(i not in valid_target_ids for i in ids):
            issues.append(f"matching_results contains invalid target ID for {sid}")
        match_map_check[sid] = ids

    for sid in s1_ids:
        if sid not in cand_map:
            issues.append(f"missing S1 in candidate_pairs: {sid}")
        if sid not in match_map_check:
            issues.append(f"missing S1 in matching_results: {sid}")
        else:
            extra = match_map_check[sid] - cand_map.get(sid, set())
            if extra:
                issues.append(
                    f"{sid}: matched IDs are not a subset of candidates: {list(extra)[:5]}"
                )

    if issues:
        print("VALIDATION FAILED")
        for x in issues[:30]:
            print("-", x)
        print("Total issues:", len(issues))
        raise AssertionError("Submission validation failed.")

    n_candidate_s1 = sum(bool(v) for v in cand_map.values())
    n_match_s1 = sum(bool(v) for v in match_map_check.values())
    n_candidate_pairs = sum(len(v) for v in cand_map.values())
    n_match_pairs = sum(len(v) for v in match_map_check.values())

    print("="*65)
    print("SUBMISSION VALIDATION PASSED")
    print("="*65)
    print("TEST S1 rows          :", f"{len(s1_ids):,}")
    print("S1s with candidates   :", f"{n_candidate_s1:,}")
    print("S1s with matches      :", f"{n_match_s1:,}")
    print("Candidate pairs       :", f"{n_candidate_pairs:,}")
    print("Matched pairs         :", f"{n_match_pairs:,}")
    print("Matches subset cand.  : YES")
    print("Duplicate IDs         : NO")
    print("="*65)

validate_outputs()


## 8. Final files

In [ ]:
for p in [candidate_path, matching_path]:
    size_mb = p.stat().st_size / 1024**2
    print(f"{p.name:25s} {size_mb:,.2f} MB")

elapsed = time.time() - RUN_START
print()
print(f"Total runtime: {elapsed/60:.2f} minutes")

resource_check("FINAL", hard=False)

print("\nFiles ready for submission:")
print("output/candidate_pairs.tsv")
print("output/matching_results.tsv")


### Important

This notebook is intentionally **not** the old 10K LightGBM experiment.

It is a bounded, submission-oriented pipeline. It does not claim that exact matching recovers every true entity pair; its purpose is to produce a valid, precision-oriented submission without the multi-hour/OOM behavior of the previous approach.

After this notebook passes validation, copy the two TSV files into the required submission ZIP structure and include the source code/documentation required by the competition.
